# Car Rental Customer Service Assistant

A customer service assistant for a car rental company in Greece with tool calling capabilities.

## Features
- SQLite database with car models and pricing
- Dynamic tool calling for car availability and pricing
- Gradio interface for customer service chat
- Sample questions for testing
- Pythonic tool handler with dynamic function calling

## Setup and Configuration

In [ ]:
# Import dependencies
import os
import sqlite3
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

# Load environment variables
load_dotenv(override=True)

# Configuration
OLLAMA_BASE_URL = os.getenv('OLLAMA_BASE_URL', 'http://localhost:11434/v1')
OLLAMA_MODEL = os.getenv('OLLAMA_MODEL', 'gemma4:e4b')

# Initialize OpenAI client for Ollama
openai = OpenAI(api_key="ollama", base_url=OLLAMA_BASE_URL)

print("Configuration:")
print(f"   Ollama Base URL: {OLLAMA_BASE_URL}")
print(f"   Model: {OLLAMA_MODEL}")

## System Message

In [ ]:
system_message = """
You are a helpful customer service assistant for GreeceCarRental, a car rental company in Greece.
You help customers with car rentals, pricing, availability, and general inquiries.
Give short, courteous answers, no more than 1-2 sentences.
Always be accurate. If you don't know the answer, say so.
Use the available tools to check car availability and pricing when needed.
"""

print("System message defined for GreeceCarRental assistant")

## Database Setup

In [ ]:
# Database setup
DB = "car_rental.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS cars (
            car_model TEXT PRIMARY KEY,
            category TEXT,
            price_per_day REAL,
            available INTEGER
        )
    ''')
    conn.commit()

print(f"Database '{DB}' created/verified successfully")

## Tool Functions

In [ ]:
def get_car_price(car_model):
    """Get car rental price per day from database"""
    print(f"DATABASE TOOL CALLED: Getting price for {car_model}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price_per_day, category FROM cars WHERE car_model = ?', (car_model.lower(),))
        result = cursor.fetchone()
        if result:
            return f"The {car_model} ({result[1]}) costs €{result[0]} per day."
        else:
            return f"Sorry, we don't have information about {car_model} in our fleet."

def check_car_availability(car_model):
    """Check if a car is available for rent"""
    print(f"DATABASE TOOL CALLED: Checking availability for {car_model}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT available, price_per_day FROM cars WHERE car_model = ?', (car_model.lower(),))
        result = cursor.fetchone()
        if result:
            status = "available" if result[0] > 0 else "not available"
            return f"The {car_model} is {status} for rent at €{result[1]} per day."
        else:
            return f"Sorry, we don't have {car_model} in our fleet."

def list_available_cars(category=None):
    """List all available cars, optionally filtered by category"""
    print(f"DATABASE TOOL CALLED: Listing available cars (category: {category})", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        if category:
            cursor.execute('SELECT car_model, price_per_day FROM cars WHERE available > 0 AND category = ?', (category.lower(),))
        else:
            cursor.execute('SELECT car_model, price_per_day FROM cars WHERE available > 0')
        results = cursor.fetchall()
        if results:
            car_list = ", ".join([f"{row[0]} (€{row[1]}/day)" for row in results])
            return f"Available cars: {car_list}"
        else:
            return "No cars are currently available."

def set_car_availability(car_model, available):
    """Set car availability in database"""
    print(f"DATABASE TOOL CALLED: Setting availability for {car_model} to {available}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO cars (car_model, category, price_per_day, available) 
            VALUES (?, ?, 0, ?) 
            ON CONFLICT(car_model) DO UPDATE SET available = ?
        ''', (car_model.lower(), 'unknown', available, available))
        conn.commit()
    return f"Availability for {car_model} set to {available}"

def add_car_to_fleet(car_model, category, price_per_day, available=1):
    """Add a new car to the fleet"""
    print(f"DATABASE TOOL CALLED: Adding {car_model} to fleet", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('''
            INSERT INTO cars (car_model, category, price_per_day, available) 
            VALUES (?, ?, ?, ?) 
            ON CONFLICT(car_model) DO UPDATE SET category = ?, price_per_day = ?, available = ?
        ''', (car_model.lower(), category.lower(), price_per_day, available, category.lower(), price_per_day, available))
        conn.commit()
    return f"{car_model} added to fleet as {category} at €{price_per_day}/day with {available} unit(s) available."

print("Tool functions defined successfully")

## Tool Definitions for LLM

In [ ]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_car_price",
            "description": "Get the daily rental price for a specific car model.",
            "parameters": {
                "type": "object",
                "properties": {
                    "car_model": {
                        "type": "string",
                        "description": "The car model to get the price for (e.g., Toyota Corolla, BMW X5)",
                    },
                },
                "required": ["car_model"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "check_car_availability",
            "description": "Check if a specific car model is available for rent.",
            "parameters": {
                "type": "object",
                "properties": {
                    "car_model": {
                        "type": "string",
                        "description": "The car model to check availability for",
                    },
                },
                "required": ["car_model"],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_available_cars",
            "description": "List all available cars, optionally filtered by category (economy, compact, midsize, luxury, suv).",
            "parameters": {
                "type": "object",
                "properties": {
                    "category": {
                        "type": "string",
                        "description": "Optional category filter (economy, compact, midsize, luxury, suv)",
                    },
                },
                "required": [],
                "additionalProperties": False
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "add_car_to_fleet",
            "description": "Add a new car to the rental fleet or update an existing one.",
            "parameters": {
                "type": "object",
                "properties": {
                    "car_model": {
                        "type": "string",
                        "description": "The car model to add",
                    },
                    "category": {
                        "type": "string",
                        "description": "The car category (economy, compact, midsize, luxury, suv)",
                    },
                    "price_per_day": {
                        "type": "number",
                        "description": "Daily rental price in Euros",
                    },
                    "available": {
                        "type": "integer",
                        "description": "Number of units available (default 1)",
                    },
                },
                "required": ["car_model", "category", "price_per_day"],
                "additionalProperties": False
            }
        }
    }
]

print("Tool definitions created for LLM")

## Pythonic Tool Handler

In [ ]:
def handle_tool_calls(message):
    """
    Dynamically call functions by name without hardcoded if statements.
    This is the Pythonic way!
    """
    responses = []
    for tool_call in message.tool_calls:
        function_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        
        # Use globals() to dynamically look up and call the function
        func = globals().get(function_name)
        
        if func is None:
            result = f"Error: Function '{function_name}' not found"
        else:
            try:
                result = func(**arguments)
            except Exception as e:
                result = f"Error calling {function_name}: {str(e)}"
        
        responses.append({
            "role": "tool",
            "content": result,
            "tool_call_id": tool_call.id
        })
    
    return responses

print("Tool handler ready to dynamically call functions")

## Chat Function with Tool Support

In [ ]:
def chat(message, history):
    """Main chat function that handles tool calls"""
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    
    response = openai.chat.completions.create(model=OLLAMA_MODEL, messages=messages, tools=tools)
    
    # Keep looping while the LLM wants to call tools
    while response.choices[0].finish_reason == "tool_calls":
        message_obj = response.choices[0].message
        tool_responses = handle_tool_calls(message_obj)
        messages.append(message_obj)
        messages.extend(tool_responses)
        response = openai.chat.completions.create(model=OLLAMA_MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

print("Chat function with tool support ready")

## Populate Database with Test Data

In [ ]:
# Sample car fleet data
car_fleet = [
    ("Toyota Aygo", "economy", 25.0, 5),
    ("Toyota Yaris", "economy", 30.0, 4),
    ("Hyundai i20", "economy", 28.0, 3),
    ("Kia Rio", "economy", 27.0, 4),
    ("Volkswagen Polo", "compact", 35.0, 3),
    ("Toyota Corolla", "compact", 40.0, 5),
    ("Hyundai i30", "compact", 38.0, 3),
    ("Kia Ceed", "compact", 37.0, 2),
    ("Volkswagen Golf", "midsize", 45.0, 4),
    ("Toyota Camry", "midsize", 55.0, 2),
    ("BMW 3 Series", "midsize", 65.0, 2),
    ("Audi A4", "midsize", 70.0, 2),
    ("BMW X5", "suv", 85.0, 2),
    ("Audi Q7", "suv", 90.0, 1),
    ("Mercedes GLE", "suv", 95.0, 1),
    ("Mercedes S-Class", "luxury", 150.0, 1),
    ("BMW 7 Series", "luxury", 140.0, 1),
    ("Audi A8", "luxury", 145.0, 1),
]

# Populate database
for car_model, category, price, available in car_fleet:
    add_car_to_fleet(car_model, category, price, available)

print("✓ Database initialized with test data")
print(f"✓ Added {len(car_fleet)} car models to the fleet")

## Sample Questions for Testing

In [ ]:
sample_questions = [
    "What cars do you have available for rent?",
    "How much does it cost to rent a Toyota Corolla per day?",
    "Do you have any luxury cars available?",
    "Is the BMW X5 available for rent?",
    "What economy cars can I rent?",
    "How much is the Mercedes S-Class per day?",
    "Do you have any SUVs in your fleet?",
    "What's the cheapest car you have available?",
    "Can you show me all midsize cars?",
    "I need a car for 5 days, what would you recommend?",
]

print("Sample questions prepared for testing:")
for i, question in enumerate(sample_questions, 1):
    print(f"{i}. {question}")

## Launch Gradio Chat Interface

In [ ]:
print("✓ Database initialized with test data")
print("✓ Tool handler is ready to dynamically call functions")
print("✓ Launching Gradio chat interface...")

# Launch chat interface
gr.ChatInterface(
    fn=chat,
    type="messages",
    title="GreeceCarRental Customer Service",
    description="Ask about car availability, pricing, and rental information for our Greece car rental service.",
    examples=sample_questions
).launch()

## Technical Implementation Details

### Database Schema
- **car_model**: Primary key, car model name
- **category**: Car category (economy, compact, midsize, luxury, suv)
- **price_per_day**: Daily rental price in Euros
- **available**: Number of units available

### Tool Functions
- **get_car_price**: Get pricing for specific car model
- **check_car_availability**: Check if a car is available
- **list_available_cars**: List all available cars (optionally by category)
- **add_car_to_fleet**: Add/update cars in the fleet

### Tool Calling Pattern
- Dynamic function calling using `globals()`
- No hardcoded if statements
- Automatic tool response handling
- Loop until LLM stops calling tools

### Car Fleet Categories
- **Economy**: Toyota Aygo, Yaris, Hyundai i20, Kia Rio (€25-30/day)
- **Compact**: VW Polo, Toyota Corolla, Hyundai i30, Kia Ceed (€35-40/day)
- **Midsize**: VW Golf, Toyota Camry, BMW 3 Series, Audi A4 (€45-70/day)
- **SUV**: BMW X5, Audi Q7, Mercedes GLE (€85-95/day)
- **Luxury**: Mercedes S-Class, BMW 7 Series, Audi A8 (€140-150/day)

### Future Improvements
- Add booking/reservation functionality
- Implement date range availability checking
- Add customer information management
- Support for multiple rental locations
- Integration with payment processing
- Multi-language support (Greek, English)